In [ ]:
# Time-Series Feature Engineering


from pathlib import Path
import pandas as pd

# 1. Load the cleaned dataset

project_root = Path.cwd()

if not (project_root / "data" / "processed" / "omni2_cleaned.csv").exists():
    project_root = project_root.parent

data_path = project_root / "data" / "processed" / "omni2_cleaned.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {data_path}"
    )

df = pd.read_csv(data_path, parse_dates=["Timestamp"])
df = df.set_index("Timestamp").sort_index()

if df.empty:
    raise ValueError("The cleaned dataset is empty.")

if "Dst_index" not in df.columns:
    raise ValueError("The required target column 'Dst_index' is missing.")

if df.index.has_duplicates:
    raise ValueError("Duplicate timestamps were found.")

if not df.index.is_monotonic_increasing:
    raise ValueError("Timestamps must be in chronological order.")

print("Cleaned dataset loaded successfully.")
print(f"Original observations: {len(df):,}")
print(f"Date range: {df.index.min()} to {df.index.max()}")


# 2. Validate hourly timestamp continuity

time_differences = df.index.to_series().diff().dropna()

if not (time_differences == pd.Timedelta(hours=1)).all():
    raise ValueError(
        "The dataset contains missing or irregular hourly timestamps. "
        "Resolve the time gaps before creating lag features."
    )

print("Hourly timestamp continuity verified.")


# 3. Define input variables

# Use all available cleaned variables except Timestamp.
# Dst_index is included because its historical values are
# available when predicting the next hour's Dst_index.

feature_columns = list(df.columns)

if not df[feature_columns].select_dtypes(
    include="number"
).columns.equals(df[feature_columns].columns):
    raise ValueError("All input variables must be numeric.")

print(f"Current input variables: {len(feature_columns)}")


# 4. Create historical lag features

df_features = df.copy()

for column in feature_columns:
    for lag in range(1, 4):
        df_features[f"{column}_lag_{lag}h"] = (
            df[column].shift(lag)
        )

print("Lag features created for t-1, t-2, and t-3.")


# 5. Create the one-hour-ahead prediction target

df_features["Dst_target_t_plus_1"] = (
    df["Dst_index"].shift(-1)
)

print("One-hour-ahead Dst prediction target created.")


# 6. Identify required columns and handle missing values

lag_columns = [
    f"{column}_lag_{lag}h"
    for column in feature_columns
    for lag in range(1, 4)
]

required_columns = (
    feature_columns
    + lag_columns
    + ["Dst_target_t_plus_1"]
)

# Do not impute the target or historical measurements here.
# Remove rows that cannot provide a complete model example.
original_rows = len(df_features)

df_features = df_features.dropna(
    subset=required_columns
).copy()

removed_rows = original_rows - len(df_features)

if df_features.empty:
    raise ValueError(
        "No complete observations remain after removing missing values."
    )

print(f"Rows removed due to missing required values: {removed_rows:,}")
print(f"Usable engineered observations: {len(df_features):,}")


# 7. Validate target alignment against the original data

expected_target = df["Dst_index"].shift(-1)

# Compare each retained timestamp with the target calculated
# from the original chronological dataset.
actual_target = df_features["Dst_target_t_plus_1"]
expected_retained_target = expected_target.loc[df_features.index]

if not actual_target.equals(expected_retained_target):
    raise AssertionError(
        "Target alignment failed: target must be the next hour's Dst."
    )

if not df_features.index.is_unique:
    raise AssertionError("Duplicate timestamps exist in the output.")

if not df_features.index.is_monotonic_increasing:
    raise AssertionError("Output timestamps are not chronological.")

print("Target alignment verified.")
print("Chronological order and unique timestamps verified.")


# 8. Save the engineered dataset

output_path = (
    project_root / "data" / "processed" / "omni2_features.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

df_features.to_csv(
    output_path,
    index_label="Timestamp"
)

print(f"Engineered dataset saved to: {output_path}")


# 9. Display final summary

print("\n" + "=" * 55)
print("TIME-SERIES FEATURE ENGINEERING SUMMARY")
print("=" * 55)
print(f"Original observations: {len(df):,}")
print(f"Final observations: {len(df_features):,}")
print(f"Original input variables: {len(feature_columns)}")
print(f"Historical lag features: {len(lag_columns)}")
print("Prediction target: Dst_target_t_plus_1")
print(
    "Remaining missing values in required columns:",
    int(df_features[required_columns].isna().sum().sum())
)
print(f"Output file: {output_path}")

display(df_features.head())



Cleaned dataset loaded successfully.
Original observations: 52,584
Date range: 2018-01-01 00:00:00 to 2023-12-31 23:00:00
Hourly timestamp continuity verified.
Current input variables: 12
Lag features created for t-1, t-2, and t-3.
One-hour-ahead Dst prediction target created.
Rows removed due to missing required values: 4
Usable engineered observations: 52,580
Target alignment verified.
Chronological order and unique timestamps verified.
Engineered dataset saved to: c:\Users\daksh\OneDrive\Desktop\space_weather_prediction\data\processed\omni2_features.csv

TIME-SERIES FEATURE ENGINEERING SUMMARY
Original observations: 52,584
Final observations: 52,580
Original input variables: 12
Historical lag features: 36
Prediction target: Dst_target_t_plus_1
Remaining missing values in required columns: 0
Output file: c:\Users\daksh\OneDrive\Desktop\space_weather_prediction\data\processed\omni2_features.csv


,Scalar_B,BX_GSM,BY_GSM,BZ_GSM,Temperature,Proton_Density,Flow_Speed,Flow_Pressure,Electric_Field,Kp_index,...,Kp_index_lag_1h,Kp_index_lag_2h,Kp_index_lag_3h,Dst_index_lag_1h,Dst_index_lag_2h,Dst_index_lag_3h,AE_index_lag_1h,AE_index_lag_2h,AE_index_lag_3h,Dst_target_t_plus_1
Timestamp,,,,,,,,,,,,,,,,,,,,,
2018-01-01 03:00:00,10.9,2.1,9.2,-2.5,70701.0,22.4,396.0,6.22,0.99,37,...,33.0,33.0,33.0,-1.0,6.0,10.0,349.0,370.0,330.0,-7.0
2018-01-01 04:00:00,11.0,-1.3,-1.9,2.4,154791.0,19.0,416.0,5.97,-1.00,37,...,37.0,33.0,33.0,0.0,-1.0,6.0,395.0,349.0,370.0,-16.0
2018-01-01 05:00:00,8.3,4.3,1.8,-2.5,257368.0,10.1,441.0,3.67,1.10,37,...,37.0,37.0,33.0,-7.0,0.0,-1.0,225.0,395.0,349.0,-19.0
2018-01-01 06:00:00,6.1,-3.3,3.0,0.4,214287.0,9.3,429.0,3.28,-0.17,23,...,37.0,37.0,37.0,-16.0,-7.0,0.0,180.0,225.0,395.0,-17.0
2018-01-01 07:00:00,6.8,-3.3,5.0,-1.4,208866.0,9.0,436.0,3.27,0.61,23,...,23.0,37.0,37.0,-19.0,-16.0,-7.0,123.0,180.0,225.0,-16.0


In [2]:

# Reload the saved dataset and verify its basic properties.

saved_df = pd.read_csv(
    output_path,
    parse_dates=["Timestamp"],
    index_col="Timestamp"
)

assert not saved_df.empty
assert saved_df.index.is_unique
assert saved_df.index.is_monotonic_increasing
assert "Dst_target_t_plus_1" in saved_df.columns
assert saved_df[required_columns].notna().all().all()

print("Saved output validation passed.")
print(f"Saved rows: {len(saved_df):,}")
print(f"Saved columns: {len(saved_df.columns)}")
print(f"File size: {output_path.stat().st_size / 1024:.2f} KB")


Saved output validation passed.
Saved rows: 52,580
Saved columns: 49
File size: 13816.52 KB
